# Femur & Hip CT Segmentation: Validation and Femoral-Head Sizing

This notebook reproduces the whole study in the
[femur-ct-validation](https://github.com/anuujshrestha/femur-ct-validation) repository.
It runs a pretrained bone-segmentation model (**TotalSegmentator**, nnU-Net) on public CT scans that have
expert masks, in its normal (1.5 mm) and `--fast` (3 mm) versions. It then measures:

1. **segmentation accuracy**: Dice, HD95 and ASSD (in mm, using the real voxel spacing);
2. **3D reconstruction**: STL meshes of the femur;
3. **femoral-head diameter and centre** from a sphere fit, compared with the same measurement from the expert mask.

There is **no training**; this is a validation study. Results and the write-up are in `README.md`,
`results/analysis.md` and `reports/technical_report.pdf`.

**How to run:** *Runtime → Change runtime type → T4 GPU*, then *Runtime → Run all*.
About 2 minutes per CT for both models, so roughly 45 minutes for 20 CTs. Finished cases are skipped
if you re-run after a disconnect.

Set `QUICK_TEST = True` to run everything in about 2 minutes on a synthetic phantom with a fake model
(no GPU and no download). This is useful to check the code.

## 1. Settings

In [ ]:
N_CASES       = 20                # number of CT scans to evaluate
MODES         = ["full", "fast"]  # full = normal 1.5 mm model, fast = 3 mm model
MIN_VOXELS    = 5000              # each of the 4 bones needs at least this many GT voxels
EXPORT_STL    = True              # export left-femur meshes (.stl)
SAVE_TO_DRIVE = True              # copy results to Google Drive
QUICK_TEST    = False             # True: synthetic phantom + fake model, no GPU or download

import os, sys, glob, shutil, subprocess
import pandas as pd
from IPython.display import Image, Markdown, display
IN_COLAB  = "google.colab" in sys.modules
BASE      = "/content" if IN_COLAB else os.path.abspath("colab_local")
DRIVE_OUT = "/content/drive/MyDrive/femur-ct-validation-results"
os.makedirs(BASE, exist_ok=True)

## 2. Get the code and install packages
In Colab, this clones the repository. Run locally, the notebook uses the repository it sits in.

In [ ]:
if os.path.isdir("femur3d"):                      # already inside the repository
    REPO = os.getcwd()
else:
    REPO = f"{BASE}/femur-ct-validation"
    if not os.path.isdir(REPO):
        subprocess.run(["git", "clone", "-q", "https://github.com/anuujshrestha/femur-ct-validation", REPO], check=True)
os.chdir(REPO)
# Outputs. QUICK_TEST writes elsewhere so it never overwrites the real results in the repository.
RES, PREDS, CASES = (("results", "preds", "cases.csv") if not QUICK_TEST else
                     (f"{BASE}/quick/results", f"{BASE}/quick/preds", f"{BASE}/quick/cases.csv"))
print("code in", REPO, "| results ->", RES)
if IN_COLAB:
    subprocess.run("pip install -q TotalSegmentator nibabel scikit-image trimesh opencv-python-headless tabulate", shell=True)
gpu = shutil.which("nvidia-smi")
print("GPU:", subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], capture_output=True, text=True).stdout.strip()
      if gpu else "none (fine for QUICK_TEST; otherwise choose a T4 GPU runtime)")

In [ ]:
def sh(cmd, env=None):
    # Run a shell command, print its output, and stop the notebook if it fails.
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True, env=env)
    print(r.stdout[-4000:])
    if r.returncode != 0:
        print(r.stderr[-4000:]); raise RuntimeError(f"failed: {cmd}")

env = dict(os.environ)
if QUICK_TEST:   # put the fake TotalSegmentator first on PATH
    os.chmod("tests/fake_bin/TotalSegmentator", 0o755)
    env["PATH"] = f"{REPO}/tests/fake_bin:" + env["PATH"]
if IN_COLAB and SAVE_TO_DRIVE and not QUICK_TEST:
    from google.colab import drive
    drive.mount("/content/drive"); os.makedirs(DRIVE_OUT, exist_ok=True)

## 3. Self-test on a synthetic phantom
This builds a fake CT whose femoral head is an exact sphere with a **46.0 mm** diameter, and a fake model that
shrinks the true mask (normal) or grows it (fast). The measured true head must come out within 0.5 mm of 46.0;
the shrunken prediction must give a smaller head and the grown one a larger head. If this fails, stop here.

In [ ]:
ST = f"{BASE}/selftest"; shutil.rmtree(ST, ignore_errors=True)
fake_env = dict(os.environ, PATH=f"{REPO}/tests/fake_bin:" + os.environ["PATH"]); os.chmod("tests/fake_bin/TotalSegmentator", 0o755)
sh(f"python tests/make_synthetic.py {ST}/data")
sh(f"python scripts/select_cases.py {ST}/data --n 2 --min_voxels 100 --out {ST}/cases.csv")
sh(f"python scripts/run_totalseg.py {ST}/data {ST}/cases.csv --out {ST}/preds", env=fake_env)
sh(f"python scripts/evaluate.py {ST}/data {ST}/cases.csv --preds {ST}/preds --out {ST}/results")
h = pd.read_csv(f"{ST}/results/femoral_head.csv")
display(h[["image_id", "mode", "side", "gt_diam_mm", "pred_diam_mm", "diam_err_mm"]].round(2))
ok = (abs(h.gt_diam_mm.mean() - 46.0) < 0.5 and (h[h["mode"] == "full"].diam_err_mm < 0).all()
      and (h[h["mode"] == "fast"].diam_err_mm > 0).all())
print(f"GT diameter {h.gt_diam_mm.mean():.2f} mm (truth 46.0):", "SELF-TEST PASSED" if ok else "SELF-TEST FAILED")
assert ok

## 4. Data
**TotalSegmentator dataset v2.0.1, small subset**: 102 CTs with expert-reviewed masks, 3.2 GB, CC BY 4.0
([Zenodo](https://zenodo.org/records/10047263)). The data are unzipped to local disk, not to Drive.

**Note:** this subset has **no test-split scans** (98 train, 4 validation). TotalSegmentator was trained on these
scans, so its absolute accuracy here is optimistic. The comparison between the normal and fast models is still fair.

In [ ]:
if QUICK_TEST:
    DATA = f"{BASE}/synthetic"; shutil.rmtree(DATA, ignore_errors=True)
    sh(f"python tests/make_synthetic.py {DATA}"); N_CASES, MIN_VOXELS = 3, 100
else:
    DATA_ROOT, ZIP = f"{BASE}/data", f"{BASE}/ds.zip"
    URLS = ["https://www.dropbox.com/scl/fi/pee5yxebfxrhz007cbuy5/Totalsegmentator_dataset_small_v201.zip?rlkey=osvfk02jc4lw5gr6uhrldtb9e&dl=1",
            "https://zenodo.org/records/10047263/files/Totalsegmentator_dataset_small_v201.zip?download=1"]
    if not glob.glob(f"{DATA_ROOT}/**/meta.csv", recursive=True):
        if not os.path.exists(ZIP) or os.path.getsize(ZIP) < 3e9:
            for url in URLS:
                print("downloading from", url.split("/")[2])
                if subprocess.run(["wget", "-q", "--show-progress", "-O", ZIP, url]).returncode == 0 and os.path.getsize(ZIP) > 3e9:
                    break
        subprocess.run(["unzip", "-q", "-o", ZIP, "-d", DATA_ROOT], check=True)
    DATA = os.path.dirname(glob.glob(f"{DATA_ROOT}/**/meta.csv", recursive=True)[0])
first = sorted(glob.glob(f"{DATA}/s*"))[0]
print("DATA =", DATA, "| scans:", len(glob.glob(f"{DATA}/s*/ct.nii.gz")))
print(os.path.basename(first), "->", sorted(os.listdir(first)))

## 5. Select cases and run TotalSegmentator (the long step)
The first call downloads the model weights. Run times are logged to `preds/runtimes.csv`.

In [ ]:
os.makedirs(os.path.dirname(CASES) or ".", exist_ok=True)
sh(f"python scripts/select_cases.py {DATA} --n {N_CASES} --min_voxels {MIN_VOXELS} --out {CASES}")
cases = pd.read_csv(CASES); display(cases.split.value_counts())
sh(f"python scripts/run_totalseg.py {DATA} {CASES} --out {PREDS} --modes {' '.join(MODES)}", env=env)
display(pd.read_csv(f"{PREDS}/runtimes.csv").groupby("mode").seconds.describe().round(1))

## 6. Evaluate against the expert masks
Dice, HD95 and ASSD per bone; femoral-head diameter and centre per side; STL meshes.

In [ ]:
sh(f"python scripts/evaluate.py {DATA} {CASES} --preds {PREDS} --out {RES} --modes {' '.join(MODES)} {'--stl' if EXPORT_STL else ''}")

## 7. Quality checks
For every hip: **metal** near the femur (a hip replacement: more than 200 voxels above 2,500 HU), whether the
**head is cut off** by the bottom of the scan, and the details of each **sphere fit** (number of points, RMSE).
These checks were added during the study after the first implant was found; they are applied identically to every hip.

In [ ]:
sh(f"python scripts/quality_checks.py {DATA} {CASES} --preds {PREDS} --out {RES} --modes {' '.join(MODES)}")
seg, head = pd.read_csv(f"{RES}/segmentation_metrics.csv"), pd.read_csv(f"{RES}/femoral_head.csv")
warn = 0
for _, r in head[head["mode"] == MODES[0]].iterrows():
    if pd.isna(r.gt_diam_mm):
        print(f"[fit failed] {r.image_id} {r.side}"); warn += 1
    elif not 35 <= r.gt_diam_mm <= 60:
        print(f"[implausible GT diameter {r.gt_diam_mm:.1f} mm] {r.image_id} {r.side}"); warn += 1
for _, r in seg[seg.bone.str.startswith("femur")].iterrows():
    if r.dice < 0.8:
        print(f"[low Dice {r.dice:.3f}] {r.image_id} {r['mode']} {r.bone}"); warn += 1
print("sanity warnings:", warn)

## 8. Final analysis
Excludes implant and cut-off hips from head sizing, runs paired statistics, and flags head fits supported by fewer
than 2,500 surface points. **That threshold was chosen after seeing the data**, so treat it as exploratory.

In [ ]:
sh(f"python scripts/analyse.py {RES}")
display(Markdown(open(f"{RES}/analysis.md").read()))

## 9. Figures
In the overlays, green is the expert mask and red the prediction, on a coronal slice through the head. The patient's left is on the viewer's left.

In [ ]:
flags = pd.read_csv(f"{RES}/implant_flags.csv")
show = [(cases.image_id.iloc[0], "left")] + [tuple(x) for x in flags[flags.implant][["image_id", "side"]].values[:1]]
for case, side in show:
    sh(f"python scripts/make_figures.py {DATA} --case {case} --side {side} --preds {PREDS} --results {RES}")
for f in ["head_error_vs_dice", "bland_altman_head_diameter_qc", "dice_boxplot"] + [f"qc_{c}_{m}_{s}" for c, s in show for m in MODES]:
    p = f"{RES}/figures/{f}.png"
    if os.path.exists(p):
        print(p); display(Image(p, width=650))

## 10. Save results
Zips the results (without the large NIfTI predictions and STL meshes), copies everything to Drive, and downloads the zip.

In [ ]:
shutil.copy(CASES, f"{RES}/cases.csv"); shutil.copy(f"{PREDS}/runtimes.csv", f"{RES}/runtimes.csv")
ZIP_OUT = os.path.abspath(f"{RES}/../results.zip")
subprocess.run(f'cd {RES}/.. && zip -qr {ZIP_OUT} {os.path.basename(RES)} -x "*/stl/*"', shell=True, check=True)
if IN_COLAB and SAVE_TO_DRIVE and not QUICK_TEST:
    shutil.copytree(RES, f"{DRIVE_OUT}/results", dirs_exist_ok=True); shutil.copy(ZIP_OUT, DRIVE_OUT)
    print("copied to", DRIVE_OUT)
if IN_COLAB:
    from google.colab import files; files.download(ZIP_OUT)
print("done:", ZIP_OUT)